## Install zstandard if not already installed

In [ ]:
!pip install zstandard

## Import libraries

In [39]:
import requests
import os
import pandas as pd
import numpy as np

## Specify file and URL locations

In [3]:
scratch_dir = 'scratch'
base_url = 'https://directory.cms.gov'

## Define a function to construct a download URL for a given directory.cms.gov file

In [4]:
def download_url(filename):
    return '/'.join([base_url, 'downloads', filename])

## Download the manifest to find the names of the other files that are available for downloading

In [5]:
manifest_url = download_url('manifest.json')
manifest = requests.get(manifest_url).json()
files = manifest['files'].keys()

## Create a scratch folder if one does not already exist and download the available files to the scratch folder if it does not already contain all available files

In [6]:
if not os.path.exists(scratch_dir): 
    os.mkdir(scratch_dir)
if len(os.listdir(scratch_dir))<len(files):
    for file in files:
        with open(os.path.join('scratch', f'{file}.zst'), 'wb') as f:
            file_url = download_url(f'{file}.zst')
            file_content = requests.get(file_url).content
            f.write(file_content)

## Pull the first 100,000 records from each ndjson file and use that sample to generate a rough data schema

In [ ]:
df_dict = {}
columns = []
for file in files:
    resource_name = file.split('.')[0]
    chunks = pd.read_json(os.path.join(scratch_dir, f'{file}.zst'), lines=True, chunksize = 100000)
    for chunk in chunks:
        df_dict[resource_name] = chunk
        for column in chunk.columns:
            val = chunk[column][0]
            many = np.nan
            if column == 'extension':
                for extension in val:
                    extension_name = extension['url'].split('/')[-1].replace('base-ext-','').replace('-',' ')
                    value_key = [k for k in extension.keys() if 'value' in k][0]
                    columns.append({'Resource':resource_name, 'Column': f'{column} - {extension_name}', 'Many': many, 'Example': extension[value_key]})
            else:
                if isinstance(val, list):
                    val = val[0]
                    many = True
                if isinstance(val, dict):
                    for subcolumn in val.keys():
                        columns.append({'Resource':resource_name, 'Column': f'{column} - {subcolumn}', 'Many': many, 'Example': val[subcolumn]})
                else:
                    columns.append({'Resource':resource_name, 'Column': column, 'Example': val})
        break

## Export the rough schema to CSV

In [54]:
column_df = pd.DataFrame(columns)
column_df

,Resource,Column,Example,Many
0,Endpoint,resourceType,Endpoint,NaN
1,Endpoint,id,Endpoint-4415de89-9982-495e-8203-d68f62d14211,NaN
2,Endpoint,meta - lastUpdated,2026-04-09T11:48:37.471494Z,NaN
3,Endpoint,extension - endpoint rank,1,NaN
4,Endpoint,extension - verification status,{'coding': [{'system': 'http://hl7.org/fhir/us...,NaN
...,...,...,...,...
103,PractitionerRole,telecom - system,phone,True
104,PractitionerRole,telecom - value,4693417800,True
105,PractitionerRole,telecom - use,practice,True
106,PractitionerRole,period,NaN,NaN


In [57]:
column_df.to_csv('schema.csv', index=False)